# Data Layer Testing Notebook
This notebook tests data access across different layers:
- Raw data from CSV
- Bronze layer (Parquet files and DuckDB)
- Silver layer (DuckDB)

### Reading Raw Data

In [19]:
# Read raw customers data
import pandas as pd
import os

# Set up path to data_raw
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
raw_data_path = os.path.join(project_root, 'data_raw', 'customers.csv')

# Read and display customers data
try:
    df_customers_raw = pd.read_csv(raw_data_path)
    print(f"Total rows in customers.csv: {len(df_customers_raw)}")
    print("\nFirst few rows:")
    print(df_customers_raw.head())
    print("\nColumns:")
    print(df_customers_raw.columns.tolist())
except Exception as e:
    print(f"Error reading {raw_data_path}:", e)

Total rows in customers.csv: 800

First few rows:
   customer_id      natural_key first_name last_name                  email  \
0            1       CUST-}-{A8       Kara     Heath   thomas87@example.net   
1            2  CUST--9A]0{A[{A      David   Ramirez   steven64@example.net   
2            3  CUST--A]09]8{{8  Alejandro    Chavez  romancody@example.org   
3            4          CUST-A[      David       Lee    april76@example.com   
4            5         CUST--]{  Cassandra     Davis      creed@example.com   

             phone        address_line1  address_line2              city  \
0  +61 459 351 432     97 Joshua Little            NaN      Lambertmouth   
1   (02) 3985 2447       8 Hoffman Bend            NaN          Johnport   
2  +61.440.449.286       30 Amy Freeway            NaN      Jenniferberg   
3     (03)90463061  356 Raymond Footway            NaN     West Michelle   
4  +61-2-7024-8405   36 Patrick Cutting            NaN  St. Jenniferfurt   

  state_region  po

### Reading DuckDB data

In [20]:
import duckdb
import os
import pandas as pd
import pytz

# Timezone
utc_plus_8 = pytz.timezone('Australia/Perth')

# Path to DuckDB file
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
duckdb_path = os.path.join(project_root, "duckdb", "warehouse.duckdb")
 
# Connect to DuckDB
con = duckdb.connect(duckdb_path)
print("Connected to: ",duckdb_path)

# List tables in raw_dataset
raw_tables = con.execute("""
    SELECT *--table_name
    FROM information_schema.columns
    WHERE table_name = 'customers'
""").fetchdf()
print("Tables in 'raw_dataset:")
print(raw_tables)

# Sample query
try:
    df_customers = con.execute("SELECT * FROM bronze.customers LIMIT 10").fetchdf()
    print("\nSample from raw_dataset.customers:")
    print(df_customers)
except Exception as e:
    print("\nError querying customers:", e)

Connected to:  c:\Users\atyagi\local_de_assessment_bundle\duckdb\warehouse.duckdb
Tables in 'raw_dataset:
   table_catalog table_schema table_name    column_name  ordinal_position  \
0      warehouse       bronze  customers    customer_id                 1   
1      warehouse       bronze  customers    natural_key                 2   
2      warehouse       bronze  customers     first_name                 3   
3      warehouse       bronze  customers      last_name                 4   
4      warehouse       bronze  customers          email                 5   
5      warehouse       bronze  customers          phone                 6   
6      warehouse       bronze  customers  address_line1                 7   
7      warehouse       bronze  customers  address_line2                 8   
8      warehouse       bronze  customers           city                 9   
9      warehouse       bronze  customers   state_region                10   
10     warehouse       bronze  customers       

In [1]:
# Check loaded data in DuckDB using Python
import duckdb
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
duckdb_path = os.path.join(project_root, "duckdb", "warehouse.duckdb")
con = duckdb.connect(duckdb_path)
print ("Connected to: ", duckdb_path)

# Query row count from bronze.customers
try:
    count = con.execute("SELECT COUNT(*) FROM bronze.customers").fetchone()[0]
    print(f"Row count in bronze.customers: {count}")
except Exception as e:
    print("Error querying bronze.customers:", e)


Connected to:  c:\Users\atyagi\local_de_assessment_bundle\duckdb\warehouse.duckdb
Row count in bronze.customers: 800


### Reading Bronze Layer Data (Parquet and DuckDB)

In [21]:
# Method 1: Reading from Parquet files
import pandas as pd
import os

# Set up paths
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
parquet_path = os.path.join(project_root, 'lake/bronze/parquet/bronze/customers')

# Read all parquet files in the directory
try:
    df_parquet = pd.read_parquet(parquet_path)
    print("=== Data from Parquet files ===")
    print(f"Total rows: {len(df_parquet)}")
    print("\nFirst 5 rows:")
    print(df_parquet.head())
except Exception as e:
    print("Error reading parquet files:", e)

# Method 2: Reading from DuckDB
import duckdb

# Connect to DuckDB
duckdb_path = os.path.join(project_root, "duckdb", "warehouse.duckdb")
con = duckdb.connect(duckdb_path)

try:
    print("\n=== Data from DuckDB ===")
    # Get total count
    count = con.execute("SELECT COUNT(*) FROM bronze.customers").fetchone()[0]
    print(f"Total rows: {count}")
    
    # Get sample data
    df_duckdb = con.execute("""
        SELECT *
        FROM bronze.customers
        LIMIT 5
    """).fetchdf()
    
    print("\nFirst 5 rows:")
    print(df_duckdb)
    
except Exception as e:
    print("Error querying DuckDB:", e)
finally:
    con.close()

=== Data from Parquet files ===
Total rows: 800

First 5 rows:
   customer_id      natural_key first_name last_name                  email  \
0            1       CUST-}-{A8       Kara     Heath   thomas87@example.net   
1            2  CUST--9A]0{A[{A      David   Ramirez   steven64@example.net   
2            3  CUST--A]09]8{{8  Alejandro    Chavez  romancody@example.org   
3            4          CUST-A[      David       Lee    april76@example.com   
4            5         CUST--]{  Cassandra     Davis      creed@example.com   

             phone        address_line1 address_line2              city  \
0  +61 459 351 432     97 Joshua Little                    Lambertmouth   
1   (02) 3985 2447       8 Hoffman Bend                        Johnport   
2  +61.440.449.286       30 Amy Freeway                    Jenniferberg   
3     (03)90463061  356 Raymond Footway                   West Michelle   
4  +61-2-7024-8405   36 Patrick Cutting                St. Jenniferfurt   

  state_reg

### Reading Silver Layer Data
Now we'll read the transformed data from the silver layer in DuckDB

# Important: Run dbt Models First
Before reading the transformed data, we need to run our dbt models to create the tables:

```bash
cd dbt
dbt run --select stg_customers
```

This will create the `stg_customers` table in the `bronze_stg` schema.

In [26]:
# Connect to DuckDB
import duckdb
import os
import pandas as pd
import subprocess

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
duckdb_path = os.path.join(project_root, "duckdb", "warehouse.duckdb")
con = duckdb.connect(duckdb_path)

try:
    print("=== Silver Layer Data (bronze_stg schema) ===")
    
    # First, let's see what tables are in the bronze_stg schema
    silver_tables = con.execute("""
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema = 'bronze_stg'
    """).fetchdf()
    print("\nTables in bronze_stg schema:")
    print(silver_tables)
    
    if len(silver_tables) == 0:
        print("\nNo tables found in bronze_stg schema. Running dbt model...")
        # Change to dbt directory and run the model
        original_dir = os.getcwd()
        try:
            os.chdir(os.path.join(project_root, "dbt"))
            subprocess.run(["dbt", "run", "--select", "stg_customers"], check=True)
            print("\nDBT model run complete. Checking tables again...")
            
            # Check tables again after running dbt
            silver_tables = con.execute("""
                SELECT table_name 
                FROM information_schema.tables 
                WHERE table_schema = 'bronze_stg'
            """).fetchdf()
            print("\nUpdated tables in bronze_stg schema:")
            print(silver_tables)
        finally:
            os.chdir(original_dir)
    
    # Read from stg_customers
    print("\n=== Data from stg_customers ===")
    # Get total count
    count = con.execute("SELECT COUNT(*) FROM bronze_stg.stg_customers").fetchone()[0]
    print(f"Total rows: {count}")
    
    # Get sample data with quality checks
    df_silver = con.execute("""
        SELECT 
            customer_id,
            email,
            first_name,
            last_name,
            country_code,
            quality_checks
        FROM bronze_stg.stg_customers
        LIMIT 5
    """).fetchdf()
    
    print("\nFirst 5 rows from silver layer:")
    print(df_silver)
    
    # Show records with quality issues
    print("\n=== Records with Quality Issues ===")
    df_quality_issues = con.execute("""
        SELECT 
            customer_id,
            email,
            country_code,
            quality_checks
        FROM bronze_stg.stg_customers
        WHERE quality_checks IS NOT NULL
        LIMIT 5
    """).fetchdf()
    
    print("\nSample records with quality issues:")
    print(df_quality_issues)
    
except Exception as e:
    print("Error querying silver layer:", e)
    print("\nPlease ensure that:")
    print("1. You have activated your Python virtual environment")
    print("2. dbt is properly installed and configured")
    print("3. You have run 'dbt run --select stg_customers' from the dbt directory")
finally:
    con.close()

=== Silver Layer Data (bronze_stg schema) ===

Tables in bronze_stg schema:
Empty DataFrame
Columns: [table_name]
Index: []

No tables found in bronze_stg schema. Running dbt model...

DBT model run complete. Checking tables again...

Updated tables in bronze_stg schema:
Empty DataFrame
Columns: [table_name]
Index: []

=== Data from stg_customers ===
Error querying silver layer: Catalog Error: Table with name stg_customers does not exist!
Did you mean "bronze.customers"?

LINE 1: SELECT COUNT(*) FROM bronze_stg.stg_customers
                             ^

Please ensure that:
1. You have activated your Python virtual environment
2. dbt is properly installed and configured
3. You have run 'dbt run --select stg_customers' from the dbt directory

DBT model run complete. Checking tables again...

Updated tables in bronze_stg schema:
Empty DataFrame
Columns: [table_name]
Index: []

=== Data from stg_customers ===
Error querying silver layer: Catalog Error: Table with name stg_customers does

### Checking Table Location in DuckDB
Let's verify where the stg_customers table is stored in DuckDB

In [32]:
# Check table location in DuckDB
import duckdb
import os
import subprocess

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
duckdb_path = os.path.join(project_root, "duckdb", "warehouse.duckdb")
con = duckdb.connect(duckdb_path)

try:
    # Check all tables in the database
    print("=== Tables in DuckDB ===")
    tables = con.execute("""
        SELECT table_schema, table_name 
        FROM information_schema.tables 
        --WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
        --ORDER BY table_schema, table_name
    """).fetchdf()
    print(tables)
    
    # Check if stg_customers exists
    stg_exists = con.execute("""
        SELECT COUNT(*) 
        FROM information_schema.tables 
        WHERE table_schema = 'bronze_stg' 
        AND table_name = 'stg_customers'
    """).fetchone()[0]
    
    if not stg_exists:
        print("\nstg_customers table not found. Running dbt model...")
        original_dir = os.getcwd()
        try:
            os.chdir(os.path.join(project_root, "dbt"))
            subprocess.run(["dbt", "run", "--select", "stg_customers"], check=True)
            print("DBT model run complete.")
        finally:
            os.chdir(original_dir)
    
    # Check stg_customers specifically
    print("\n=== stg_customers Details ===")
    columns = con.execute("""
        SELECT column_name, data_type, is_nullable
        FROM information_schema.columns
        WHERE table_schema = 'bronze_stg' 
        AND table_name = 'stg_customers'
        ORDER BY ordinal_position
    """).fetchdf()
    print(columns)
    
    # Get table statistics
    print("\n=== Table Statistics ===")
    stats = con.execute("""
        SELECT 
            COUNT(*) as total_rows,
            COUNT(*) FILTER (WHERE quality_checks IS NOT NULL) as rows_with_issues,
            COUNT(DISTINCT country_code) as unique_countries,
            MIN(join_ts_utc) as earliest_join_date,
            MAX(join_ts_utc) as latest_join_date
        FROM bronze_stg.stg_customers
    """).fetchdf()
    print(stats)
    
except Exception as e:
    print("Error querying database:", e)
    print("\nTroubleshooting steps:")
    print("1. Make sure you're in the dbt project directory")
    print("2. Run 'dbt run --select stg_customers'")
    print("3. Check dbt logs for any errors")
    print("4. Verify the schema name is 'bronze_stg' in dbt_project.yml")
finally:
    con.close()

=== Tables in DuckDB ===
      table_schema           table_name
0           bronze            customers
1           bronze       exchange_rates
2           bronze        orders_header
3           bronze         orders_lines
4           bronze             products
5           bronze              returns
6           bronze         returns_base
7           bronze      returns_evolved
8           bronze       returns_upsert
9           bronze              sensors
10          bronze            shipments
11          bronze               stores
12          bronze         stores__null
13          bronze            suppliers
14          bronze      suppliers__null
15          bronze           _dlt_loads
16          bronze  _dlt_pipeline_state
17          bronze         _dlt_version
18  bronze_staging        orders_header
19  bronze_staging         orders_lines
20  bronze_staging              returns
21  bronze_staging            shipments
22  bronze_staging         _dlt_version

stg_customers 

### Verify DBT Model Creation
Let's check if the dbt models are being created correctly and troubleshoot any issues

In [ ]:
# Detailed verification of dbt model creation
import duckdb
import os
import subprocess

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
duckdb_path = os.path.join(project_root, "duckdb", "warehouse.duckdb")

print("=== DBT Environment Check ===")
print(f"Project root: {project_root}")
print(f"DuckDB path: {duckdb_path}")
print(f"DuckDB file exists: {os.path.exists(duckdb_path)}")

# Connect to DuckDB
con = duckdb.connect(duckdb_path)

try:
    # 1. Check all schemas
    print("\n=== All Schemas ===")
    schemas = con.execute("""
        SELECT DISTINCT schema_name 
        FROM information_schema.schemata 
        ORDER BY schema_name
    """).fetchdf()
    print(schemas)
    
    # 2. Check if bronze_stg schema exists
    print("\n=== bronze_stg Schema Check ===")
    bronze_stg_exists = con.execute("""
        SELECT COUNT(*) 
        FROM information_schema.schemata 
        WHERE schema_name = 'bronze_stg'
    """).fetchone()[0]
    print(f"bronze_stg schema exists: {bool(bronze_stg_exists)}")
    
    # 3. List all tables in all schemas
    print("\n=== All Tables by Schema ===")
    all_tables = con.execute("""
        SELECT table_schema, table_name, table_type
        FROM information_schema.tables 
        WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
        ORDER BY table_schema, table_name
    """).fetchdf()
    print(all_tables)
    
    # 4. Try to create schema manually if it doesn't exist
    if not bronze_stg_exists:
        print("\nTrying to create bronze_stg schema...")
        con.execute("CREATE SCHEMA IF NOT EXISTS bronze_stg")
        print("Schema created.")
    
    # 5. Run dbt model again
    print("\n=== Running DBT Model ===")
    original_dir = os.getcwd()
    try:
        os.chdir(os.path.join(project_root, "dbt"))
        result = subprocess.run(["dbt", "run", "--select", "stg_stores"], 
                              capture_output=True, text=True, check=False)
        print("DBT Output:")
        print(result.stdout)
        if result.stderr:
            print("DBT Errors:")
            print(result.stderr)
    finally:
        os.chdir(original_dir)
    
    # 6. Verify again after dbt run
    print("\n=== Final Table Verification ===")
    final_check = con.execute("""
        SELECT table_schema, table_name 
        FROM information_schema.tables 
        WHERE table_schema = 'bronze_stg'
        ORDER BY table_name
    """).fetchdf()
    print(final_check)
    
except Exception as e:
    print(f"\nError during verification: {e}")
finally:
    con.close()

In [35]:
import duckdb
import pandas as pd
import pytz
import os
from datetime import datetime
 
# Timezone
utc_plus_8 = pytz.timezone('Australia/Perth')
 
# Path to DuckDB file
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
duckdb_path = os.path.join(project_root, "duckdb", "warehouse.duckdb")
 
# Connect to DuckDB
con = duckdb.connect(duckdb_path)
 
# List tables in raw_dataset
raw_tables = con.execute("""
    SELECT table_schema,table_name
    FROM information_schema.tables
    --WHERE table_schema = 'bronze_stg'
""").fetchdf()
print("Tables in 'raw_dataset':")
print(raw_tables)
 
 
 

Tables in 'raw_dataset':
      table_schema           table_name
0           bronze            customers
1           bronze       exchange_rates
2           bronze        orders_header
3           bronze         orders_lines
4           bronze             products
5           bronze              returns
6           bronze         returns_base
7           bronze      returns_evolved
8           bronze       returns_upsert
9           bronze              sensors
10          bronze            shipments
11          bronze               stores
12          bronze         stores__null
13          bronze            suppliers
14          bronze      suppliers__null
15          bronze           _dlt_loads
16          bronze  _dlt_pipeline_state
17          bronze         _dlt_version
18  bronze_staging        orders_header
19  bronze_staging         orders_lines
20  bronze_staging              returns
21  bronze_staging            shipments
22  bronze_staging         _dlt_version
